# Chapter 23 — Independent Calls

**Companion to Applied AI**

Question: Does issuing several calls make their results independent?

By the end of this notebook you will have:

- generated multiple attempts from a seeded simulator
- measured individual vs at-least-one success
- shown independence is an empirical property, checked — never assumed

## What this notebook demonstrates
A seeded simulator (stand-in for sampling) measuring what fan-out actually buys — including the case where branches share a hidden flaw.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

seed: 42


## 1. Independent attempts: the math that fan-out relies on

In [2]:
p, k = 0.4, 3
theory = 1 - (1 - p) ** k
print(f"p={p}, k={k}: at-least-one-success in theory = {theory:.3f}")
N = 4000
rng = random.Random(SEED)
hits = sum(1 for _ in range(N) if any(rng.random() < p for _ in range(k)))
print(f"simulated: {hits / N:.3f}")
assert abs(hits / N - theory) < 0.03

p=0.4, k=3: at-least-one-success in theory = 0.784
simulated: 0.782


## 2. Correlated branches: same calls, shared latent failure

In [3]:
def trial(shared_fail_prob: float) -> bool:
    r = random.Random()
    # must reseed per call for determinism of the demo
    return None
rng = random.Random(SEED)
def run_batch(shared: bool) -> float:
    ok = 0
    for _ in range(N):
        if shared and rng.random() < 0.3:  # latent flaw kills all k branches
            continue
        if any(rng.random() < p for _ in range(k)):
            ok += 1
    return ok / N
ind = run_batch(False)
rng = random.Random(SEED)
cor = run_batch(True)
print(f"independent branches: {ind:.3f}   shared-flaw branches: {cor:.3f}")
assert cor < ind

independent branches: 0.782   shared-flaw branches: 0.551


## 3. Sealed fan-out: lineage that is declared can be filtered

In [4]:
SENTINEL = "SENT-secret-xyz"
branches = [{"id": f"b{i}", "lineage": ["task-1"], "prompt": f"solve part {i}"} for i in range(3)]
# a branch that copies the sentinel into its prompt defeats the seal
branches[1]["prompt"] += " " + SENTINEL
def sealed_package(bs):
    leaked = [b["id"] for b in bs if SENTINEL in b["prompt"]]
    return {"branches": [b["id"] for b in bs], "leak_in_prompt": leaked}
print(sealed_package(branches))
assert sealed_package(branches)["leak_in_prompt"] == ["b1"]

{'branches': ['b0', 'b1', 'b2'], 'leak_in_prompt': ['b1']}


## Interpretation
- Supports: `blind ≠ independent ≠ diverse`; fan-out math holds only when branches do not share failure modes; seals filter declared lineage, not copied text.
- Does NOT support: claims about any real model's sample diversity.

## Try it yourself
1. Raise the shared-flaw rate to 0.6 and watch at-least-one collapse.
2. Estimate pairwise branch agreement as a cheap empirical independence check.
3. Add per-branch lineage IDs and verify the seal drops undeclared branches.